# Combined EDA notes

This notebook contains my EDA for the six processed datasets.
I used files from `data/processed` and exported tables/plots to the `results` folders.

Datasets: iea, sales, population, charging, stock, survey.

Export paths:
- `results/tables/multi_dataset_eda/<dataset>/`
- `results/figures/multi_dataset_eda/<dataset>/`

Old `consumer` outputs are kept as archive.

In [1]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 200)

In [2]:
project_root = Path.cwd()
if project_root.name == 'notebooks':
    project_root = project_root.parent

processed_dir = project_root / 'data' / 'processed'
results_tables_root = project_root / 'results' / 'tables' / 'multi_dataset_eda'
results_figures_root = project_root / 'results' / 'figures' / 'multi_dataset_eda'

DATASETS = {
    'iea': processed_dir / 'iea_ev_clean.csv',
    'sales': processed_dir / 'ev_sales_clean.csv',
    'population': processed_dir / 'ev_population_clean.csv',
    'charging': processed_dir / 'charging_infrastructure_clean.csv',
    'stock': processed_dir / 'stock_market_clean.csv',
    'survey': processed_dir / 'survey_clean.csv',
}

for name in DATASETS:
    (results_tables_root / name).mkdir(parents=True, exist_ok=True)
    (results_figures_root / name).mkdir(parents=True, exist_ok=True)

missing_files = [str(p) for p in DATASETS.values() if not p.exists()]
if missing_files:
    raise FileNotFoundError(f'Missing processed files: {missing_files}')

DATASETS

{'iea': PosixPath('/home/ravindran/workspace/college/6th_Sem/Business Analytics/project/eVehicle-adoption-analytics/data/processed/iea_ev_clean.csv'),
 'sales': PosixPath('/home/ravindran/workspace/college/6th_Sem/Business Analytics/project/eVehicle-adoption-analytics/data/processed/ev_sales_clean.csv'),
 'population': PosixPath('/home/ravindran/workspace/college/6th_Sem/Business Analytics/project/eVehicle-adoption-analytics/data/processed/ev_population_clean.csv'),
 'charging': PosixPath('/home/ravindran/workspace/college/6th_Sem/Business Analytics/project/eVehicle-adoption-analytics/data/processed/charging_infrastructure_clean.csv'),
 'stock': PosixPath('/home/ravindran/workspace/college/6th_Sem/Business Analytics/project/eVehicle-adoption-analytics/data/processed/stock_market_clean.csv'),
 'survey': PosixPath('/home/ravindran/workspace/college/6th_Sem/Business Analytics/project/eVehicle-adoption-analytics/data/processed/survey_clean.csv')}

In [8]:
# Keep output folders clean: remove stale PNGs created in previous runs for these 6 datasets.
managed_datasets = ['iea', 'sales', 'population', 'charging', 'stock', 'survey']
allowed_per_dataset = {
    'correlation_heatmap.png',
    'numeric_distributions.png',
    'missing_values_top20.png',
    'yearly_trend.png',
}
allowed_root_files = {
    'cross_dataset_normalized_yearly_trends.png',
    'stock_ev_scatter.png',
}

removed = []
for ds in managed_datasets:
    folder = results_figures_root / ds
    if not folder.exists():
        continue
    for p in folder.glob('*.png'):
        if p.name not in allowed_per_dataset:
            p.unlink(missing_ok=True)
            removed.append(str(p))

for p in results_figures_root.glob('*.png'):
    if p.name not in allowed_root_files:
        p.unlink(missing_ok=True)
        removed.append(str(p))

print(f'Cleanup done. Removed {len(removed)} stale image(s).')

Cleanup done. Removed 0 stale image(s).


## Data loading

In [3]:
def load_datasets(dataset_map):
    loaded = {}
    for name, path in dataset_map.items():
        df = pd.read_csv(path)
        df.columns = [c.strip() for c in df.columns]
        loaded[name] = df
    return loaded

dfs = load_datasets(DATASETS)
{k: v.shape for k, v in dfs.items()}

{'iea': (2975, 5),
 'sales': (16436, 3),
 'population': (714, 5),
 'charging': (38219, 4),
 'stock': (3356, 9),
 'survey': (325, 41)}

## Dataset profiles and summary tables

In [4]:
def profile_dataset(df, dataset_name, table_root):
    out_dir = table_root / dataset_name
    out_dir.mkdir(parents=True, exist_ok=True)

    profile = pd.DataFrame({
        'dataset': [dataset_name],
        'rows': [len(df)],
        'columns': [df.shape[1]],
        'duplicate_rows': [int(df.duplicated().sum())],
        'missing_cells': [int(df.isna().sum().sum())],
    })
    profile.to_csv(out_dir / 'profile.csv', index=False)

    col_overview = pd.DataFrame({
        'column': df.columns,
        'dtype': [str(df[c].dtype) for c in df.columns],
        'non_null': [int(df[c].notna().sum()) for c in df.columns],
        'null_count': [int(df[c].isna().sum()) for c in df.columns],
        'null_pct': [float(df[c].isna().mean() * 100) for c in df.columns],
        'n_unique': [int(df[c].nunique(dropna=True)) for c in df.columns],
    })
    col_overview.to_csv(out_dir / 'column_overview.csv', index=False)

    num_cols = df.select_dtypes(include=np.number).columns.tolist()
    if num_cols:
        num_summary = df[num_cols].describe().T.reset_index().rename(columns={'index': 'column'})
        num_summary.to_csv(out_dir / 'numeric_summary.csv', index=False)

        corr = df[num_cols].corr(numeric_only=True)
        corr.to_csv(out_dir / 'correlation_matrix.csv')

        out_rows = []
        for c in num_cols:
            s = df[c].dropna()
            if s.empty:
                continue
            q1, q3 = s.quantile(0.25), s.quantile(0.75)
            iqr = q3 - q1
            if iqr == 0:
                out_count = 0
            else:
                low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
                out_count = int(((s < low) | (s > high)).sum())
            out_rows.append({'column': c, 'q1': q1, 'q3': q3, 'iqr': iqr, 'outlier_count': out_count})
        pd.DataFrame(out_rows).to_csv(out_dir / 'outlier_summary_iqr.csv', index=False)

    cat_cols = [c for c in df.columns if c not in num_cols]
    if cat_cols:
        cat_rows = []
        for c in cat_cols:
            s = df[c]
            vc = s.value_counts(dropna=False)
            cat_rows.append({
                'column': c,
                'n_unique': int(s.nunique(dropna=True)),
                'mode': vc.index[0] if len(vc) else np.nan,
                'mode_count': int(vc.iloc[0]) if len(vc) else 0,
            })
            top10 = s.fillna('<NA>').astype(str).value_counts().head(10).reset_index()
            top10.columns = ['value', 'count']
            safe_col = c.lower().replace(' ', '_').replace('/', '_').replace('-', '_')
            top10.to_csv(out_dir / f'top10_{safe_col}.csv', index=False)

        pd.DataFrame(cat_rows).to_csv(out_dir / 'categorical_summary.csv', index=False)

    return profile

profiles = [profile_dataset(df, name, results_tables_root) for name, df in dfs.items()]
master_profile_df = pd.concat(profiles, ignore_index=True)
master_profile_df.to_csv(results_tables_root / 'master_dataset_profile.csv', index=False)
master_profile_df

,dataset,rows,columns,duplicate_rows,missing_cells
0,iea,2975,5,261,0
1,sales,16436,3,2037,0
2,population,714,5,0,0
3,charging,38219,4,158,0
4,stock,3356,9,0,0
5,survey,325,41,3,1803


## Core plots

In [5]:
def plot_missing_top20(df, dataset_name, fig_root):
    miss = (df.isna().mean() * 100).sort_values(ascending=False).head(20)
    if miss.sum() == 0:
        return
    plt.figure(figsize=(10, 6))
    sns.barplot(x=miss.values, y=miss.index, hue=miss.index, legend=False, palette='Reds_r')
    plt.title(f'{dataset_name.upper()} - Top Missing Columns (%)')
    plt.xlabel('Missing %')
    plt.ylabel('Column')
    plt.tight_layout()
    plt.savefig(fig_root / dataset_name / 'missing_values_top20.png', dpi=200)
    plt.close()

def plot_numeric_distributions(df, dataset_name, fig_root, max_cols=8):
    num_cols = df.select_dtypes(include=np.number).columns.tolist()
    if not num_cols:
        return
    cols = num_cols[:max_cols]
    n = len(cols)
    ncols = 2
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(12, 4 * nrows))
    axes = np.array(axes).reshape(-1)
    for i, c in enumerate(cols):
        sns.histplot(df[c].dropna(), kde=True, ax=axes[i], color='steelblue')
        axes[i].set_title(c)
    for j in range(i + 1, len(axes)):
        axes[j].axis('off')
    fig.suptitle(f'{dataset_name.upper()} - Numeric Distributions', y=1.01)
    plt.tight_layout()
    plt.savefig(fig_root / dataset_name / 'numeric_distributions.png', dpi=200, bbox_inches='tight')
    plt.close()

def plot_correlation_heatmap(df, dataset_name, fig_root):
    num_cols = df.select_dtypes(include=np.number).columns.tolist()
    if len(num_cols) < 2:
        return
    corr = df[num_cols].corr(numeric_only=True)
    plt.figure(figsize=(9, 7))
    sns.heatmap(corr, cmap='coolwarm', center=0, linewidths=0.3)
    plt.title(f'{dataset_name.upper()} - Correlation Heatmap')
    plt.tight_layout()
    plt.savefig(fig_root / dataset_name / 'correlation_heatmap.png', dpi=220)
    plt.close()

for name, df in dfs.items():
    plot_missing_top20(df, name, results_figures_root)
    plot_numeric_distributions(df, name, results_figures_root)
    plot_correlation_heatmap(df, name, results_figures_root)

print('Core figures exported.')

Core figures exported.


## Time trends and EV-stock check

In [7]:
def detect_time_col(df):
    candidates = ['year', 'model year', 'date', 'Date']
    lower_map = {c.lower(): c for c in df.columns}
    for c in candidates:
        if c.lower() in lower_map:
            return lower_map[c.lower()]
    return None

def detect_primary_value_col(df):
    preferred = ['value', 'ev_sales', 'vehicle_count', 'charging_points', 'Close', 'ev_adoption_score', 'ev_readiness_score']
    for p in preferred:
        if p in df.columns and pd.api.types.is_numeric_dtype(df[p]):
            return p
    num_cols = df.select_dtypes(include=np.number).columns.tolist()
    return num_cols[0] if num_cols else None

yearly_series = []

for name, df in dfs.items():
    time_col = detect_time_col(df)
    value_col = detect_primary_value_col(df)
    if time_col is None or value_col is None:
        continue

    temp = df.copy()
    if 'date' in time_col.lower():
        temp['year'] = pd.to_datetime(temp[time_col], errors='coerce', utc=True).dt.year
    else:
        temp['year'] = pd.to_numeric(temp[time_col], errors='coerce')

    trend = temp.dropna(subset=['year', value_col]).groupby('year', as_index=False)[value_col].sum().sort_values('year')
    if trend.empty:
        continue

    trend['yoy_pct'] = trend[value_col].pct_change() * 100
    trend.to_csv(results_tables_root / name / 'yearly_trend.csv', index=False)

    plt.figure(figsize=(10, 5))
    sns.lineplot(data=trend, x='year', y=value_col, marker='o')
    plt.title(f'{name.upper()} - Yearly Trend ({value_col})')
    plt.tight_layout()
    plt.savefig(results_figures_root / name / 'yearly_trend.png', dpi=220)
    plt.close()

    if trend[value_col].nunique() > 1:
        vmin, vmax = trend[value_col].min(), trend[value_col].max()
        trend['normalized_value'] = (trend[value_col] - vmin) / (vmax - vmin)
        trend['dataset'] = name
        yearly_series.append(trend[['year', 'dataset', 'normalized_value']])

if yearly_series:
    combined = pd.concat(yearly_series, ignore_index=True)
    combined.to_csv(results_tables_root / 'cross_dataset_normalized_yearly_trends.csv', index=False)

    plt.figure(figsize=(11, 6))
    sns.lineplot(data=combined, x='year', y='normalized_value', hue='dataset', marker='o')
    plt.title('Cross-Dataset Normalized Yearly Trends')
    plt.tight_layout()
    plt.savefig(results_figures_root / 'cross_dataset_normalized_yearly_trends.png', dpi=220)
    plt.close()

# EV linkage (stock vs EV sales)
sales = dfs['sales'].copy()
sales['year'] = pd.to_numeric(sales['year'], errors='coerce')
if 'region_country' in sales.columns:
    world_sales = sales[sales['region_country'].astype(str).str.lower() == 'world'].copy()
    if world_sales.empty:
        world_sales = sales.copy()
else:
    world_sales = sales.copy()
ev_signal = world_sales.groupby('year', as_index=False)['ev_sales'].sum()

stock = dfs['stock'].copy()
stock['Date'] = pd.to_datetime(stock['Date'], errors='coerce', utc=True)
stock['year'] = stock['Date'].dt.year
stock_yearly = stock.groupby(['year', 'Company'], as_index=False)['Close'].mean()

link = stock_yearly.merge(ev_signal, on='year', how='inner')
link.to_csv(results_tables_root / 'stock_ev_linkage_yearly.csv', index=False)

if not link.empty:
    plt.figure(figsize=(10, 6))
    sns.scatterplot(data=link, x='ev_sales', y='Close', hue='Company')
    plt.title('EV Sales vs Avg Stock Close (Yearly)')
    plt.tight_layout()
    plt.savefig(results_figures_root / 'stock_ev_scatter.png', dpi=220)
    plt.close()

print('Trend and linkage outputs exported.')

Trend and linkage outputs exported.


In [10]:
# Quick evidence block for final write-up (values pulled from current run)
insights = {}

# 1) IEA trend snapshot
iea = dfs['iea'].copy()
iea['year'] = pd.to_numeric(iea['year'], errors='coerce')
latest_iea_year = int(iea['year'].max())
iea_latest = iea[iea['year'] == latest_iea_year].copy()
iea_by_param = iea_latest.groupby('parameter', as_index=False)['value'].sum().sort_values('value', ascending=False)
insights['iea_latest_year'] = latest_iea_year
insights['iea_top_parameter'] = iea_by_param.iloc[0].to_dict() if not iea_by_param.empty else None

# 2) EV sales concentration
sales_df = dfs['sales'].copy()
sales_df['year'] = pd.to_numeric(sales_df['year'], errors='coerce')
latest_sales_year = int(sales_df['year'].max())
sales_latest = sales_df[sales_df['year'] == latest_sales_year].copy()
sales_latest = sales_latest.groupby('region_country', as_index=False)['ev_sales'].sum().sort_values('ev_sales', ascending=False)
sales_latest_no_world = sales_latest[sales_latest['region_country'].str.lower() != 'world'].copy()
top3_sales = sales_latest_no_world.head(3)
top3_share = (top3_sales['ev_sales'].sum() / sales_latest_no_world['ev_sales'].sum()) * 100 if len(sales_latest_no_world) else np.nan
insights['sales_latest_year'] = latest_sales_year
insights['sales_top3_share_pct'] = float(top3_share) if pd.notna(top3_share) else np.nan

# 3) Population mix
pop = dfs['population'].copy()
bev_mask = pop['electric vehicle type'].str.contains('BEV', case=False, na=False)
phev_mask = pop['electric vehicle type'].str.contains('PHEV|Plug-in', case=False, na=False)
total_vehicle_count = pop['vehicle_count'].sum()
bev_count = pop.loc[bev_mask, 'vehicle_count'].sum()
phev_count = pop.loc[phev_mask, 'vehicle_count'].sum()
insights['bev_share_pct'] = float((bev_count / total_vehicle_count) * 100) if total_vehicle_count else np.nan
insights['phev_share_pct'] = float((phev_count / total_vehicle_count) * 100) if total_vehicle_count else np.nan

# 4) Charging concentration
chg = dfs['charging'].copy()
country_chg = chg.groupby('country', as_index=False)['charging_points'].sum().sort_values('charging_points', ascending=False)
top5_chg_share = (country_chg.head(5)['charging_points'].sum() / country_chg['charging_points'].sum()) * 100 if len(country_chg) else np.nan
insights['charging_top5_share_pct'] = float(top5_chg_share) if pd.notna(top5_chg_share) else np.nan

# 5) Survey readiness and barriers
survey = dfs['survey'].copy()
barrier_cols = [
    'evs_are_too_expensive',
    'charging_infrastructure_is_insufficient',
    'battery_replacement_cost_is_high',
    'driving_range_is_inadequate',
    'charging_time_is_too_long',
    'ev_resale_value_is_uncertain',
    'limited_service_centers',
]
survey_barrier_means = survey[barrier_cols].apply(pd.to_numeric, errors='coerce').mean().sort_values(ascending=False)
insights['survey_top_barrier'] = survey_barrier_means.index[0]
insights['survey_top_barrier_mean'] = float(survey_barrier_means.iloc[0])

# 6) Stock-EV linkage (association only)
if not link.empty:
    corr_by_company = link.groupby('Company')[['ev_sales', 'Close']].corr().reset_index()
    corr_by_company = corr_by_company[corr_by_company['level_1'] == 'ev_sales'][['Company', 'Close']]
    corr_by_company = corr_by_company.rename(columns={'Close': 'corr_ev_sales_close'})
    corr_by_company = corr_by_company.sort_values('corr_ev_sales_close', ascending=False).reset_index(drop=True)
else:
    corr_by_company = pd.DataFrame(columns=['Company', 'corr_ev_sales_close'])

# 7) Data quality flags
dup_df = master_profile_df[['dataset', 'duplicate_rows']].sort_values('duplicate_rows', ascending=False).reset_index(drop=True)
miss_df = master_profile_df[['dataset', 'missing_cells']].sort_values('missing_cells', ascending=False).reset_index(drop=True)

print('Evidence table ready.')
summary_numbers = pd.DataFrame([
    {
        'latest_iea_year': insights['iea_latest_year'],
        'latest_sales_year': insights['sales_latest_year'],
        'sales_top3_share_pct': round(insights['sales_top3_share_pct'], 2),
        'bev_share_pct': round(insights['bev_share_pct'], 2),
        'phev_share_pct': round(insights['phev_share_pct'], 2),
        'charging_top5_share_pct': round(insights['charging_top5_share_pct'], 2),
        'top_barrier': insights['survey_top_barrier'],
        'top_barrier_mean': round(insights['survey_top_barrier_mean'], 2),
    }
])
summary_numbers

Evidence table ready.


,latest_iea_year,latest_sales_year,sales_top3_share_pct,bev_share_pct,phev_share_pct,charging_top5_share_pct,top_barrier,top_barrier_mean
0,2035,2030,83.92,79.95,20.05,80.34,battery_replacement_cost_is_high,4.05


## Final insights

- In the IEA data, the latest year is 2035, and EV stock is the largest metric there.
- In EV sales (latest year 2030), China is the top contributor.
- Sales are concentrated: top 3 regions together are about 84% of non-World sales.
- In population data, BEV share (~80%) is much higher than PHEV share (~20%).
- Charging points are also concentrated: top 5 countries hold about 80% of total points.
- Survey results show adoption is still cautious overall.
- The strongest barrier in the survey is battery replacement cost.
- EV sales and stock close prices move in the same direction for major EV-linked companies in this sample.
- This is correlation only, not causation.
- Data quality note: duplicates are present in sales, iea, and charging files.
- Missing values are mostly in the survey file.
- Overall trend is positive for EV growth, but cost concerns and charging access are still key barriers.